# Brain Tumor Detection with YOLOv8 & MLflow (Kaggle Pipeline)

Dataset: `/kaggle/input/datasets/anaselkhadiri2964/brain-scan-yolo/raw2`

### Complete 8-Step Pipeline:
1. **Install & Imports** (`ultralytics`, `mlflow`, `cv2`, `yaml`, `matplotlib`)
2. **Paths & MLflow Setup** (Kaggle `/kaggle/working/mlruns`)
3. **Visualize Ground-Truth Bounding Box** (Check YOLO normalized format)
4. **Train / Val / Test Split** (Organize into `train`, `val`, and `test`)
5. **Generate `data.yaml`** (Configured with `train`, `val`, `test`)
6. **Train with MLflow Tracking** (Log params, loss, metrics, and artifacts)
7. **Evaluate Metrics** (Validation vs. Unseen Test Generalization)
8. **Inference & Save Best Weights** (`/kaggle/working/yolov8_brain_tumor.pt`)

### Step 1: Install & Imports

In [ ]:
# Install ultralytics and mlflow (quiet mode)
!pip install -q ultralytics mlflow

import os
import shutil
import glob
import cv2
import yaml
import matplotlib.pyplot as plt

# Set MLflow environment flag before importing/configuring mlflow
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow
from ultralytics import YOLO, settings

# Disable background auto-logging so our explicit MLflow run has full control
settings.update({"mlflow": False})

print("Setup completed successfully!")

### Step 2: Paths & MLflow Experiment Setup

> **Why FileStore on Kaggle?**
> On Linux/Kaggle, `sqlite:///mlflow.db` (3 slashes) resolves to root `/mlflow.db`, causing a Permission Denied error.
> Using `/kaggle/working/mlruns` with `MLFLOW_ALLOW_FILE_STORE=true` avoids database locking and permission issues entirely.

In [ ]:
# 1. Dataset & working paths
RAW_DIR = "/kaggle/input/datasets/anaselkhadiri2964/brain-scan-yolo/raw2"
if not os.path.exists(RAW_DIR):
    RAW_DIR = "/kaggle/input/brain-scan-yolo/raw2"

YOLO_DIR = "/kaggle/working/yolo"
OUTPUT_DIR = "/kaggle/working"

# 2. MLflow Tracking setup (Kaggle-friendly file-store)
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

MLFLOW_URI = "/kaggle/working/mlruns" if os.path.exists("/kaggle") else "./mlruns"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("Brain_Tumor_Detection")

print("Dataset Path:       ", RAW_DIR)
print("YOLO Working Path:  ", YOLO_DIR)
print("MLflow Tracking URI:", MLFLOW_URI)

### Step 3: Visualize Sample Annotated Image

> **YOLO Normalized Box Coordinates**:
> - $x_1 = (x_{center} - \frac{width}{2}) \times W$
> - $y_1 = (y_{center} - \frac{height}{2}) \times H$
> - $x_2 = (x_{center} + \frac{width}{2}) \times W$
> - $y_2 = (y_{center} + \frac{height}{2}) \times H$

In [ ]:
# Find a sample image and its matching label
sample_img = f"{RAW_DIR}/Train/Glioma/images/gg (1).jpg"
sample_lbl = f"{RAW_DIR}/Train/Glioma/labels/gg (1).txt"

if not os.path.exists(sample_img):
    sample_img = glob.glob(f"{RAW_DIR}/Train/Glioma/images/*.*")[0]
    base = os.path.splitext(os.path.basename(sample_img))[0]
    sample_lbl = f"{RAW_DIR}/Train/Glioma/labels/{base}.txt"

img = cv2.imread(sample_img)
h, w, _ = img.shape

# Draw bounding box
with open(sample_lbl, "r") as f:
    for line in f:
        cls_id, xc, yc, bw, bh = map(float, line.strip().split())
        x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
        x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"Class {int(cls_id)}", (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Sample Annotated MRI")
plt.axis("off")
plt.show()

### Step 4: Dataset Splitting (Train, Val, Test)

Organizes the dataset into:
- **Train**: Training data (~70%)
- **Validation**: Validation during training (~15%)
- **Test**: Independent evaluation of generalization (~15%)

In [ ]:
# 1. Create target directories for train, val, and test
for split in ["train", "val", "test"]:
    os.makedirs(f"{YOLO_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{YOLO_DIR}/labels/{split}", exist_ok=True)

classes = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]

for cls in classes:
    # A. Train split
    train_img_dir = f"{RAW_DIR}/Train/{cls}/images"
    train_lbl_dir = f"{RAW_DIR}/Train/{cls}/labels"
    if os.path.exists(train_img_dir):
        for file in os.listdir(train_img_dir):
            base, _ = os.path.splitext(file)
            lbl = f"{base}.txt"
            if os.path.exists(os.path.join(train_lbl_dir, lbl)):
                shutil.copy(os.path.join(train_img_dir, file), f"{YOLO_DIR}/images/train/{file}")
                shutil.copy(os.path.join(train_lbl_dir, lbl), f"{YOLO_DIR}/labels/train/{lbl}")

    # B. Val & Test splits (Split Val 50/50 for Val and Test)
    val_img_dir = f"{RAW_DIR}/Val/{cls}/images"
    val_lbl_dir = f"{RAW_DIR}/Val/{cls}/labels"
    if os.path.exists(val_img_dir):
        val_files = [f for f in os.listdir(val_img_dir) if os.path.exists(os.path.join(val_lbl_dir, f"{os.path.splitext(f)[0]}.txt"))]
        mid = len(val_files) // 2

        # First half -> val
        for file in val_files[:mid]:
            base = os.path.splitext(file)[0]
            shutil.copy(os.path.join(val_img_dir, file), f"{YOLO_DIR}/images/val/{file}")
            shutil.copy(os.path.join(val_lbl_dir, f"{base}.txt"), f"{YOLO_DIR}/labels/val/{base}.txt")

        # Second half -> test
        for file in val_files[mid:]:
            base = os.path.splitext(file)[0]
            shutil.copy(os.path.join(val_img_dir, file), f"{YOLO_DIR}/images/test/{file}")
            shutil.copy(os.path.join(val_lbl_dir, f"{base}.txt"), f"{YOLO_DIR}/labels/test/{base}.txt")

print(f"Train images: {len(os.listdir(f'{YOLO_DIR}/images/train'))}")
print(f"Val images:   {len(os.listdir(f'{YOLO_DIR}/images/val'))}")
print(f"Test images:  {len(os.listdir(f'{YOLO_DIR}/images/test'))}")

### Step 5: Generate `data.yaml` Configuration

Defines paths to `train`, `val`, and `test` splits.

In [ ]:
data_yaml = {
    "path": YOLO_DIR,
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 4,
    "names": ["glioma", "meningioma", "no_tumor", "pituitary"]
}

yaml_path = f"{YOLO_DIR}/data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml saved at {yaml_path}")

### Step 6: Train YOLOv8 with MLflow Tracking

Tracks:
- Hyperparameters (epochs, batch size, image size, optimizer)
- Validation & Test metrics (mAP@0.5, mAP@0.5:0.95, Precision, Recall)
- Artifacts (best weights `best.pt`, confusion matrix, loss curves)

In [ ]:
# 1. Load model
model = YOLO("yolov8n.pt")

# 2. Train with MLflow tracking context
with mlflow.start_run(run_name="yolov8n_training"):
    # Log hyperparameters
    mlflow.log_params({
        "model": "yolov8n",
        "epochs": 10,
        "batch_size": 16,
        "imgsz": 640,
        "dataset": "brain_scan_yolo"
    })

    # Train model on Kaggle GPU
    results = model.train(
        data=yaml_path,
        epochs=10,
        imgsz=640,
        batch=16,
        plots=True
    )

    # 3. Evaluate and log validation metrics
    val_metrics = model.val(split="val")
    mlflow.log_metrics({
        "val_map50": float(val_metrics.box.map50),
        "val_map50_95": float(val_metrics.box.map),
        "val_precision": float(val_metrics.box.mp),
        "val_recall": float(val_metrics.box.mr)
    })

    # 4. Evaluate and log test metrics (generalization)
    test_metrics = model.val(split="test")
    mlflow.log_metrics({
        "test_map50": float(test_metrics.box.map50),
        "test_map50_95": float(test_metrics.box.map),
        "test_precision": float(test_metrics.box.mp),
        "test_recall": float(test_metrics.box.mr)
    })

    # 5. Log best weights and plots as artifacts
    best_weights = model.trainer.best
    mlflow.log_artifact(best_weights, artifact_path="weights")
    if os.path.exists(model.trainer.save_dir):
        mlflow.log_artifacts(model.trainer.save_dir, artifact_path="plots")

print("Training & MLflow logging completed successfully!")

### Step 7: Display Evaluation Metrics Summary

In [ ]:
print("=== Validation Split Metrics ===")
print("mAP@0.5:     ", val_metrics.box.map50)
print("mAP@0.5:0.95:", val_metrics.box.map)
print("Precision:   ", val_metrics.box.mp)
print("Recall:      ", val_metrics.box.mr)

print("\n=== Test Split Metrics (Generalization) ===")
print("mAP@0.5:     ", test_metrics.box.map50)
print("mAP@0.5:0.95:", test_metrics.box.map)
print("Precision:   ", test_metrics.box.mp)
print("Recall:      ", test_metrics.box.mr)

### Step 8: Inference on Unseen Test Image & Save Weights

In [ ]:
# 1. Run prediction on a test image
test_images = glob.glob(f"{YOLO_DIR}/images/test/*.jpg")
sample_test = test_images[0]

preds = model.predict(sample_test)

# 2. Display prediction plot
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(preds[0].plot(), cv2.COLOR_BGR2RGB))
plt.title("Test Image Detection Prediction")
plt.axis("off")
plt.show()

# 3. Save weights directly to /kaggle/working for download
best_model_path = f"{OUTPUT_DIR}/yolov8_brain_tumor.pt"
shutil.copy(model.trainer.best, best_model_path)

print(f"Best model saved successfully to: {best_model_path}")